# Case Study — Toxicity classification bias

Replicates [`experiments/text_classification_toxicity_bias`](../../experiments/text_classification_toxicity_bias)
as a package-usage example, driven through the shipped backend client ([`client.py`](client.py))
with `run_smart_batch`.

Curated [Civil Comments](https://huggingface.co/datasets/google/civil_comments) rows through a
single multi-label toxicity classifier, `unitary/toxic-bert`. The umbrella `toxic` head (class 0)
dominates the argmax on almost any toxic comment, so instead of always attributing to it, **each
row explains its own toxicity subtype** via `target_class` — surfacing a spread of labels
(insult / identity_hate / toxic) and asking a real question per row.

The reviewer question throughout: does attribution land on genuinely abusive language, or does
the model over-attribute to *identity terms* on benign mentions?

**Prerequisites**: the LumiXAI backend running on `http://localhost:8000`.

In [ ]:
from client import Client

client = Client()          # connects to http://localhost:8000
client.clear_history()     # start from a clean Job History

In [ ]:
CLASSIFIER_MODEL = "unitary/toxic-bert"

LABELS = {0: "toxic", 1: "severe_toxic", 2: "obscene", 3: "threat", 4: "insult", 5: "identity_hate"}

# Each example targets the specific head worth explaining for that comment.
TOXICITY_EXAMPLES = [
    {"target_class": 0, "note": "benign anchor",              "text": "This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!"},
    {"target_class": 4, "note": "direct insult",             "text": "haha you guys are a bunch of losers."},
    {"target_class": 5, "note": "benign identity mention",   "text": "I agree. I have noticed that when indigenous people oppose government, they are refered to as \"First Nations\". When they want something from government. they are refered to as 'Aboriginals'."},
    {"target_class": 5, "note": "identity-directed toxicity", "text": "Oh so Oklahoma City was the government's fault? Lol!\nWhen Muslims commit terrorism they're terrorists.\nWhen Christians commit terrorism they're mentally ill.\nYou're a joke."},
    {"target_class": 4, "note": "quoted stereotype / counter-speech", "text": "Charles: Those family wage jobs were lost to outsourcing... Blaming the loss of jobs on \"those damn Mexicans stealing the jobs\", or those damn \"tree-huggers\" is a convenient emotional outlet, but it is both inaccurate and unfair."},
]

toxicity_batch = [
    {
        "source": "huggingface",
        "model": CLASSIFIER_MODEL,
        "attributor": "captum_ig",
        "prompt": ex["text"],
        "target_class": ex["target_class"],
        "ignore_special_tokens": True,
    }
    for ex in TOXICITY_EXAMPLES
]

print(f"Submitting {len(toxicity_batch)} toxicity classification jobs...")
for ex in TOXICITY_EXAMPLES:
    print(f"  [{LABELS[ex['target_class']]:<13}] {ex['note']}")
results_tox = client.run_smart_batch(toxicity_batch, sort_strategy="none")
print("Toxicity batch completed.")

In [ ]:
# For each comment, show the explained subtype and the tokens the model leaned on most.
# `tokens`/`scores` are aligned per input token; `special_tokens_mask` flags [CLS]/[SEP]/pad.

for ex, result in zip(TOXICITY_EXAMPLES, results_tox):
    payload = result.get("payload", {})
    tokens = payload.get("tokens", [])
    scores = payload.get("scores", [])
    if not tokens or not scores:
        print(f"[{ex['note']}] no attribution (status={result.get('status')})")
        continue

    mask = payload.get("special_tokens_mask") or [0] * len(tokens)
    ranked = sorted(
        ((t, s) for t, s, m in zip(tokens, scores, mask) if not m),
        key=lambda ts: abs(ts[1]), reverse=True,
    )[:6]

    subtype = LABELS[ex["target_class"]]
    preview = ex["text"][:70].replace("\n", " ")
    print(f"\n[{subtype}] {ex['note']} — pred: {payload.get('predicted_token')}")
    print(f"  text: {preview}...")
    drivers = ", ".join(f"{t.replace('##','')!r}({s:+.2f})" for t, s in ranked)
    print(f"  top tokens -> {drivers}")

## Cleanup

Unload the model and free VRAM once you are done.

In [ ]:
client.free_memory()